# Notebook to download forex exchange rates

Latest version: 2024-11-30  
Author: MvS

## Description

Notebook to extract Bundesbank forex data for specific dates:

- use case is the conversion of foreign currency transactional income/loss (dividend, interest, trades) to the base account currency for tax purposes


## Result




In [ ]:
import requests
import pandas as pd
from io import StringIO
from pathlib import Path

# Base Bundesbank URL
BASE_URL = "https://www.bundesbank.de/statistic-rmi/StatisticDownload?tsId=BBEX3.D.{symbol}.BB.AC.000&its_csvFormat=de&mode=its"

# Function to fetch exchange rates from Bundesbank
def fetch_rates(symbol, dates):
    url = BASE_URL.format(symbol=symbol)
    print(url)
    # Fetch the CSV data from the Bundesbank API
    response = requests.get(url)
    response.raise_for_status()  # Ensure the request was successful

    # Save the CSV content to a pandas DataFrame
    data = pd.read_csv(
        StringIO(response.text), engine='python', delimiter=';', decimal=",",
        skiprows=5, skipfooter=2, usecols=[0, 1], names=["Date", "Rate"]
    )
    data["Date"] = pd.to_datetime(data["Date"], format="ISO8601")
    data["Rate"] = pd.to_numeric(data["Rate"], errors="coerce")

    # Interpolate weekend gaps with last value
    data["Rate"] = data["Rate"].ffill()
    # Drop remaining empties
    data = data.dropna()

    # Filter the data for the provided dates
    #print(data.set_index("Date").reindex(dates))["Rate"]
    return data.set_index("Date").reindex(dates)["Rate"]

# Main function to update the CSV file
def update_exchange_rates(file_path):
    # Read the CSV file
    file = Path(Path.cwd().parent, file_path)
    file = file.resolve()
    if not file.exists():
        raise FileNotFoundError(f"File not found: {file_path}")
    
    df = pd.read_csv(file)
    df["date"] = pd.to_datetime(df["date"])  # Ensure dates are in datetime format
    df = df.set_index("date")

    # Fetch and update rates for each column except 'date'
    for column in df.columns:
        symbol = column
        print(f"Fetching rates for {symbol}...")
        df[column] = fetch_rates(symbol, df.index)
        #print(df[column])
        
    
    # Write the updated DataFrame back to the file
    df.to_csv(file, index=True)
    print(f"Updated file saved to {file}")

# Call the function with the CSV file path
update_exchange_rates("data/bb-forex-rates.csv")
